# Bài 14 · Kể chuyện bằng dữ liệu và thẩm định phân tích AI

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học.** Sau notebook này, bạn sẽ:

1. Thực hiện **quy trình thẩm định bốn bước** (truy số → phương pháp → diễn giải → phán quyết)
   trên một báo cáo do AI viết.
2. Viết lại kết luận sai bằng cách diễn đạt đúng mức với số liệu.
3. Tổng hợp kết quả thành **bảng phán quyết** và đoạn tóm tắt theo cấu trúc kim tự tháp.

## Báo cáo cần thẩm định: thị trường Airbnb Santiago

Đoạn dưới là báo cáo do một chatbot viết từ dữ liệu Santiago, mốc chụp ngày 29/06/2026,
với yêu cầu *"viết báo cáo thị trường ngắn, ấn tượng"*. **Nhiệm vụ: thẩm định cả năm kết luận.**

---

> ### Báo cáo thị trường Airbnb Santiago — 06/2026
>
> **KL1.** Thị trường có **18.534 chỗ ở**, trong đó **81% là nguyên căn** — nguồn cung
> nghiêng hẳn về cho thuê cả nhà.
>
> **KL2.** Giá thuê **trung bình 118.200 CLP/đêm** (~3,3 triệu đồng) — du khách nên chuẩn bị
> ngân sách tương ứng.
>
> **KL3.** Số đánh giá **tháng 6/2026 giảm 27%** so với tháng 5 — thị trường đang
> **hạ nhiệt đáng lo ngại**.
>
> **KL4.** Chỗ ở có tên nhắc đến metro **rẻ hơn khoảng 11%** — cho thấy **vị trí gần metro
> làm giảm giá cho thuê**.
>
> **KL5.** Trong các quận có ≥500 chỗ ở, **Lo Barnechea đắt nhất** với giá trung vị
> **426.230 CLP/đêm**, bỏ xa quận thứ hai (Las Condes, ~97.000).

---

Với từng kết luận: **(1) truy số** bằng cách tự tính lại; **(2) kiểm phương pháp** như lựa chọn
trung bình hay trung vị, mùa vụ và chất lượng dữ liệu; **(3) kiểm xem diễn đạt có đúng mức**; **(4) phán quyết**.

In [ ]:
import pandas as pd
import numpy as np

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29"
df = pd.read_csv(f"{BASE}/visualisations/listings.csv")
rv = pd.read_csv(f"{BASE}/visualisations/reviews.csv", parse_dates=["date"])
print(df.shape, rv.shape)

## KL1 — "18.534 chỗ ở, 81% nguyên căn"

In [ ]:
# Bước 1: truy số
so_listing = len(df)
ty_le_nguyen_can = (df["room_type"] == "Entire home/apt").mean()
print(f"Số chỗ ở: {so_listing:,} | nguyên căn: {ty_le_nguyen_can:.1%}")

**Phán quyết KL1:** ✅ số đúng (18.534; 81,0%), phương pháp hợp lệ, diễn đạt đúng mức. **Giữ nguyên.**
Xác nhận một kết luận đúng cũng là kết quả của quá trình thẩm định.

## KL2 — "trung bình 118.200 CLP/đêm, du khách chuẩn bị ngân sách tương ứng"

In [ ]:
# Bước 1: truy số — trung bình có đúng 118.200 không?
gia = df.loc[df["price"] > 0, "price"]
print(f"Trung bình: {df['price'].mean():,.0f}")
print(f"Trung vị  : {df['price'].median():,.0f}")
print(f"P99        : {gia.quantile(0.99):,.0f}  (lớn nhất: {gia.max():,.0f})")

In [ ]:
# Bước 2: kiểm phương pháp — trung bình nhạy với giá ngoại lai đến mức nào?
mean_sach = gia[gia <= gia.quantile(0.99)].mean()
print(f"Trung bình sau khi bỏ 1% đuôi: {mean_sach:,.0f}  (trung bình gốc cao hơn {df['price'].mean()/mean_sach - 1:.0%})")

**Phán quyết KL2:** số đúng ✓ nhưng phương pháp không phù hợp với mục đích dự trù ngân sách.
Một phần trăm giá ngoại lai làm trung bình cao hơn khoảng 44%; một nửa số chỗ ở có giá dưới 59.000 CLP.

✍️ **Viết lại cho đúng mực:** *"Một nửa số chỗ ở có giá dưới 59.000 CLP/đêm (khoảng 1,65 triệu đồng);
mức trung bình 118.200 bị một nhóm nhỏ chỗ ở giá bất thường kéo lên, không phản ánh
ngân sách điển hình."*

## KL3 — "tháng 6 giảm 27% so với tháng 5 → hạ nhiệt đáng lo ngại"

In [ ]:
# Bước 1: truy số — có đúng -27%?
thang = rv[rv["date"] < "2026-07-01"].set_index("date").resample("ME").size()
mom = thang.pct_change().iloc[-1]
print(f"Tháng 6 so với tháng 5: {mom:.1%}")

# So cùng kỳ (buổi 8) kể chuyện khác hẳn:
yoy = thang.iloc[-1] / thang.iloc[-13] - 1
print(f"Tháng 6/2026 so với tháng 6/2025: {yoy:+.1%}")

# Nhưng khoan — tháng 6 các năm TRƯỚC có giảm so với tháng 5 không?
for nam in [2023, 2024, 2025]:
    t5 = thang[f"{nam}-05"].iloc[0]; t6 = thang[f"{nam}-06"].iloc[0]
    print(f"{nam}: T6 so với T5 = {t6/t5-1:+.1%}")

Trong các năm trước, số đánh giá tháng 6 **không** giảm. Trước khi diễn giải mức giảm 27%,
cần kiểm tra thước đo: dữ liệu được chụp ngày 29/06 nên khách lưu trú cuối tháng có thể chưa kịp
viết đánh giá. Vì vậy, tháng cuối của mỗi mốc chụp đều thiếu dữ liệu một cách có hệ thống.

Có thể kiểm tra hiện tượng này bằng cách so cùng tháng 9/2025 ở hai mốc chụp khác nhau:

In [ ]:
# Bằng chứng dữ liệu kỳ cuối chưa đầy đủ: so tháng 9/2025 ở hai mốc chụp
r9 = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                 "2025-09-27/visualisations/reviews.csv", parse_dates=["date"])
for nhan, r in [("mốc chụp 09/2025", r9), ("mốc chụp 06/2026 (9 tháng sau)", rv)]:
    m = r.set_index("date").resample("ME").size()
    print(f"{nhan:35} T9/2025 = {m.get(pd.Timestamp('2025-09-30'), 0):,}")

Ở mốc chụp 09/2025, tháng 9/2025 có **14.264** đánh giá; chín tháng sau, con số này là
**17.301** (+21%) vì các đánh giá viết muộn đã được bổ sung. Ngược lại, số đánh giá ở một số
tháng cũ có thể giảm khi chỗ ở rời nền tảng và các đánh giá tương ứng không còn trong dữ liệu.

**Phán quyết KL3:** số −27% đúng ✓ nhưng tháng cuối của mốc chụp **thiếu dữ liệu có hệ thống**,
nên chưa đo được mức thay đổi thực; so cùng kỳ vẫn tăng 7%. Kết luận "hạ nhiệt đáng lo ngại"
**chưa thể kiểm chứng từ một mốc chụp**.

✍️ **Viết lại:** *"Số đánh giá tháng gần nhất chưa phản ánh đủ (khách chưa kịp viết); so với
cùng kỳ 2025, số đánh giá vẫn tăng 7%. Cần mốc chụp tiếp theo để đánh giá xu hướng quý 2/2026."*

## KL4 — "gần metro làm giảm giá cho thuê"

In [ ]:
# Bước 1: truy số
gan_metro = df["name"].str.contains("metro", case=False, na=False).rename("tên nhắc metro")
chenh = df.loc[gan_metro, "price"].median() / df.loc[~gan_metro, "price"].median() - 1
print(f"Chênh lệch trung vị: {chenh:+.1%}")

# Bước 3: kiểm diễn giải — biến ẩn "loại phòng"?
print()
print(pd.crosstab(gan_metro, df["room_type"], normalize="index").round(2))

In [ ]:
# So sánh TRONG TỪNG loại phòng và kiểm tra cỡ mẫu
trung_vi = df.groupby([df["room_type"], gan_metro])["price"].median().unstack()
so_luong = df.groupby([df["room_type"], gan_metro]).size().unstack(fill_value=0)
so_trong_nhom = pd.DataFrame({
    "trung vị không nhắc": trung_vi[False],
    "trung vị có nhắc": trung_vi[True],
    "chênh %": (trung_vi[True] / trung_vi[False] - 1) * 100,
    "n không nhắc": so_luong[False],
    "n có nhắc": so_luong[True],
})
so_trong_nhom.round(1)

Nhóm có tên nhắc đến metro có tỷ trọng nguyên căn cao hơn: 91,5% so với 79,0%. Trong hai loại
phổ biến nhất, chênh lệch là 17,2% với nguyên căn (n=2.707) và 17,8% với phòng riêng (n=242).
Như vậy, loại phòng không giải thích chênh lệch quan sát được mà còn che bớt một phần chênh lệch đó.

**Phán quyết KL4:** chênh lệch vẫn tồn tại sau khi kiểm soát loại phòng ✓, nhưng kết luận nhân quả
vẫn **sai logic**. Tên có nhắc đến metro là tín hiệu do chủ nhà tự chọn, không phải thước đo
khoảng cách; ngoài ra còn nhiều biến gây nhiễu như chất lượng nội thất, tầng và tuổi chỗ ở.
Dữ liệu quan sát cho phép nói hai yếu tố "đi kèm", không cho phép nói yếu tố này "làm giảm" yếu tố kia.

✍️ **Viết lại:** *"Chỗ ở có tên nhắc đến 'metro' rẻ hơn 10,8% trên toàn bộ mẫu; trong hai loại
phòng phổ biến nhất, mức chênh lệch là khoảng 17–18%.
Để xác định tác động nhân quả của vị trí, cần đo khoảng cách thực tế đến ga và dùng một thiết kế
phân tích phù hợp."*

## KL5 — "Lo Barnechea đắt nhất trong các quận ≥500 chỗ ở, 426.230 CLP"

In [ ]:
# Bước 1 và 2: truy số với đúng điều kiện đề bài (n >= 500)
tk = df.groupby("neighbourhood")["price"].agg(trung_vi="median", n="size")
tk[tk["n"] >= 500].nlargest(3, "trung_vi").round(0)

**Phán quyết KL5:** ✅ đúng cả số lẫn điều kiện lọc. **Giữ nguyên** — có thể bổ sung n=824
để người đọc tự cân nhắc độ tin cậy.

## Tổng kết cuộc thẩm định

In [ ]:
verdict = pd.DataFrame([
    ["KL1: 18.534 chỗ ở, 81% nguyên căn", "✓", "✓", "✓", "GIỮ NGUYÊN"],
    ["KL2: trung bình 118.200 ~ ngân sách",  "✓", "✗ ngoại lai", "✗", "SỬA: dùng trung vị"],
    ["KL3: giảm 27% theo tháng ~ hạ nhiệt",  "✓", "✗ kỳ cuối thiếu", "✗", "CHƯA THỂ KIỂM CHỨNG từ một mốc"],
    ["KL4: metro làm giảm giá",             "✓", "△ đo gián tiếp", "✗ nhân quả", "SỬA: 'đi kèm'"],
    ["KL5: Lo Barnechea 426k",              "✓", "✓", "✓", "GIỮ NGUYÊN, thêm cỡ mẫu"],
], columns=["Kết luận", "Số", "Phương pháp", "Diễn giải", "Phán quyết"])
verdict

Trong báo cáo này, **AI tính đúng cả năm kết luận**. Các lỗi nằm ở phương pháp và diễn giải.
Vì vậy, người thẩm định cần hiểu dữ liệu, chứ không chỉ kiểm tra phép tính.

## Bài tập tại lớp

### Bài 1 — Viết đoạn mở đầu kim tự tháp

Từ bảng phán quyết, viết **năm câu** mở đầu báo cáo thẩm định theo cấu trúc: câu trả lời chung →
hai bằng chứng quan trọng nhất → giới hạn. Viết vào ô Markdown dưới. Gợi ý câu đầu: *"Ba trong năm
kết luận của báo cáo AI cần sửa hoặc đảo ngược, dù mọi con số đều tính đúng."*)

*(Đoạn của bạn — nhấp đúp để viết)*

### Bài 2 — Bẫy mốc so sánh

Tạo một kết luận phóng đại nhưng đúng số từ dữ liệu đánh giá bằng cách chọn mốc so sánh có lợi
(gợi ý: đáy COVID-19 năm 2020 hoặc một tháng thấp điểm). Sau đó, viết lại kết luận với mốc
so sánh hợp lý và diễn đạt đúng mức.

In [ ]:
# TODO Bài 2 (mã khởi đầu):
t_2020 = thang.loc["2020"].min()
t_moi = thang.iloc[-1]
print(f"Quá mức  : 'Đánh giá tăng {t_moi/t_2020:.0f} LẦN so với 2020!'")

t_2019 = thang.loc["2019"].mean()
print(f"Trung thực: 'Đánh giá gấp ~{t_moi/t_2019:.1f} lần mức trước dịch (TB 2019).'")

> **Đọc kỹ thước đo:** Mức 51 lần được tính bằng cách so tháng thấp nhất trong đại dịch COVID-19 (4/2020, chỉ 321 đánh giá) với tháng cuối. Do tháng cuối chưa đủ dữ liệu như KL3, bội số thực tế còn có thể cao hơn. Ở **buổi 13**, mức khoảng 16 lần được tính theo *năm* (2020→2025). Cùng một hiện tượng nhưng khác thước đo, vì vậy cần **khai báo mốc so sánh**.

### Bài 3 — Kiểm tra nghịch lý Simpson

Kiểm tra xem nghịch lý Simpson có xảy ra giữa hai mốc chụp Santiago hay không:
tính giá trung vị **toàn thị trường** và **theo từng `room_type`** cho 09/2025 và 06/2026.
Chiều nào ngược chiều nào? Tỷ trọng các loại phòng đổi ra sao?

In [ ]:
# TODO Bài 3:
t9 = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                 "2025-09-27/visualisations/listings.csv")
for ten, d in [("2025-09", t9), ("2026-06", df)]:
    print(ten, "| chung:", f"{d['price'].median():,.0f}",
          "| tỷ trọng nguyên căn:", f"{(d['room_type']=='Entire home/apt').mean():.1%}")
print()
print((pd.concat([t9.assign(s="2025-09"), df.assign(s="2026-06")])
       .groupby(["s", "room_type"])["price"].median().unstack().round(0)))

**Kết quả:** Từ 09/2025 đến 06/2026:

- Trung vị chung tăng từ **40.690** lên **59.000** CLP.
- Trung vị của mọi loại phòng đều tăng: nguyên căn 44.647→64.900; phòng riêng 25.736→34.235; phòng chung 17.464→20.541; phòng khách sạn 63.435→117.138.
- Tỷ trọng nguyên căn gần như không đổi: 80,1%→81,0%.

Như vậy, dữ liệu này **không có nghịch lý Simpson**. Bảng trong slide dùng số minh hoạ để giải thích cơ chế; cần kiểm tra cơ cấu nhóm trước khi diễn giải một số liệu gộp.

## Bài tập về nhà — Thẩm định báo cáo của nhóm

1. Lấy phần báo cáo bài tập lớn nhóm bạn đã viết hoặc nhờ một chatbot viết năm kết luận từ bảng chỉ số
   của nhóm.
2. Thực hiện quy trình thẩm định bốn bước cho **từng kết luận** và lập bảng phán quyết như trên.
3. Xác định kết luận cần sửa, cập nhật báo cáo và ghi vào mục "AI sai ở đâu" trong `AI_USAGE.md`
   (nếu lỗi do AI) — nội dung này dùng trực tiếp cho vấn đáp buổi 15.

---

## Tóm tắt buổi học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| Trình bày theo cấu trúc kim tự tháp: kết luận trước, quá trình ở phụ lục | Giúp người đọc nhanh chóng nắm được câu trả lời và bằng chứng |
| Thẩm định bốn bước: truy số → phương pháp → diễn giải → phán quyết | Là quy trình dùng trong vấn đáp bài tập lớn |
| AI có thể tính đúng nhưng chọn sai phương pháp hoặc diễn giải sai | Cần kiểm tra cả phép tính, thước đo và lập luận |
| Mốc so sánh, mùa vụ, cơ cấu mẫu và lời quá mức | Bốn nhóm rủi ro cần tự kiểm tra trước khi nộp |

**Toàn bộ nội dung môn học kết thúc ở đây.** Buổi 15 dành cho vấn đáp bài tập lớn; hãy xem
slide hướng dẫn và danh sách kiểm tra trước khi nộp bài.